# Среда и конфигурация

Все данные синтетические. Работа выполняется в отдельной ветке; эталон v2 не изменяется.

Найдём корень проекта. В Colab при отсутствии файлов клонируется только рабочая ветка; локально используются уже полученные исходники.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd()
while root != root.parent and not (root/'efc').exists():
    root = root.parent
if not (root/'efc').exists():
    import subprocess
    target = Path('/content/Forecast_Edge_Fog_Cloud')
    subprocess.run(['git','clone','--branch','night/fefc-v3','https://github.com/annfridreksen/Forecast_Edge_Fog_Cloud.git',str(target)],check=True)
    root=target
os.chdir(root)
sys.path.insert(0,str(root))
print('Рабочая папка:',root)


Что исследуем: как поток проходит Edge, отдельную FIFO одного из 50 Fog и Cloud. Интервалы из TASK.md — синтетические профили, а сетевые законы и деление Edge — явные допущения.

In [ ]:
from efc.experiments import configuration_series
from efc.model import simulate,Config,COMPONENTS
from efc.views import architecture,configuration_plots,animate_task
import pandas as pd
frame=configuration_series()
display(frame)

На схемах показаны уровни, шесть компонент и входы/выходы. Схема содержит группы устройств, но модель создаёт каждый Fog с собственной очередью.

In [ ]:
architecture()

Сравним конфигурации на одинаковых правилах. Стресс-серия 1200 событий/с отделена от парковочной нагрузки 400 событий/ч; по ней нельзя заявлять параметры реальной парковки.

In [ ]:
queue=pd.read_csv('results/stress_queue.csv')
configuration_plots(frame,queue)

Анимация одного события использует его рассчитанные интервалы, а не произвольную длительность. Базовая очередь может быть пустой — это также результат.

In [ ]:
r=simulate()
html=animate_task(r['tasks'][0])
Path('results/event_animation.html').write_text(html)

Ползунки изменяют параметры нового запуска. Если ipywidgets отсутствует в локальной среде, сначала установите зависимости; статические результаты выше уже воспроизводимы.

In [ ]:
try:
    import ipywidgets as w
    def explore(sensors=1000,fog=50,buffer=50,mobile=0.3):
        m=simulate(Config(n_edge=sensors,n_fog=fog,buffer=buffer,mobile_fraction=mobile))['metrics']
        print({k:m[k] for k in ('mean_ms','p95_ms','mean_fog_wait_ms','dropped')})
    display(w.interactive(explore,sensors=w.IntSlider(value=1000,min=500,max=2000,step=250),fog=w.IntSlider(value=50,min=25,max=100,step=25),buffer=w.IntSlider(value=50,min=0,max=400,step=10),mobile=w.FloatSlider(value=.3,min=0,max=1,step=.1)))
except ImportError:
    print('ipywidgets не установлен: интерактивные ползунки не проверены в этой среде.')

Проверка: баланс событий и сумма шести компонент. Таблица чисел предназначена для прямого переноса в статью с указанием источника.

In [ ]:
assert (frame.completed+frame.dropped==frame.arrivals).all()
for row in r['tasks']:
    if row['status']=='completed':
        assert abs(sum(row[k] for k in COMPONENTS)-row['latency_ms'])<1e-6
b=frame.query("scenario=='baseline'").iloc[0]
display(pd.DataFrame({'число в статье':[b.mean_ms,b.p95_ms,b.mean_fog_wait_ms],'число модели':[r['metrics']['mean_ms'],r['metrics']['p95_ms'],r['metrics']['mean_fog_wait_ms']]},index=['средняя,мс','p95,мс','Fog-ожидание,мс']))
assert abs(b.mean_ms-r['metrics']['mean_ms'])<1e-9
print('OK')